In [ ]:
# Lab type: review
# Course: AI403 — Building Production AI Agents
# Lesson: Tool Design Is the Real Lever: Schemas, Errors, and What a Call Returns
# Task: An AI assistant generated the tool set below for a refund agent. Judge each tool's
# schema, errors and return value, then redesign the ones that will fail in production.

# Lab: Reviewing a Generated Tool Set

The model only sees a tool's **name**, **description**, **input schema** and **what it returns**. You'll review all four for a refund agent's tools, test the schemas against calls recorded from real traces, and redesign the refund and billing tools.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
import json, re

# Nimbus Analytics billing API: what the backend returns for customer C-1042 (abridged)
BILLING_API = {
    "customer": {"id": "C-1042", "name": "Acme Corp", "created": "2023-02-11T09:14:00Z",
                 "billing_email": "ap@acme.example", "tax_id": "GB123456789",
                 "address": {"line1": "1 High St", "city": "Leeds", "postcode": "LS1 1AA"},
                 "metadata": {"crm_id": "0015g00000XyZ", "segment": "mid-market"}},
    "invoices": [
        {"id": "INV-2291", "status": "paid", "currency": "usd", "amount_due": 480000,
         "amount_paid": 480000, "period_start": "2026-03-03", "period_end": "2027-03-02",
         "lines": [{"description": "Teams plan (annual), 20 seats", "amount": 480000,
                    "tax_rates": [], "proration": False}],
         "payment_intent": "pi_3Pq8xY2eZvKYlo2C1", "hosted_url": "https://pay.example/i/2291"},
        {"id": "INV-2140", "status": "paid", "currency": "usd", "amount_due": 36000,
         "amount_paid": 36000, "period_start": "2026-01-01", "period_end": "2026-01-31",
         "lines": [{"description": "Extra seats (monthly), 10 seats", "amount": 36000,
                    "tax_rates": [], "proration": True}],
         "payment_intent": "pi_3Pn1aB2eZvKYlo2C9", "hosted_url": "https://pay.example/i/2140"},
    ],
    "credit_notes": [
        {"id": "CN-0310", "invoice": "INV-2291", "amount": 120000, "reason": "plan_downgrade",
         "created": "2026-06-14T16:02:00Z", "memo": "Downgrade Teams 20 -> 15 seats"},
    ],
}

def tokens(obj):
    return len(json.dumps(obj)) // 4        # rough: ~4 characters per token

print("raw payload:", tokens(BILLING_API), "tokens")

In [ ]:
# A small JSON-Schema checker covering the keywords used in this lab
def validate(args, schema):
    """Return a list of problems with a tool call's arguments (empty list = valid)."""
    problems = []
    props = schema.get("properties", {})
    for name in schema.get("required", []):
        if name not in args:
            problems.append(f"missing required '{name}'")
    if schema.get("additionalProperties") is False:
        problems += [f"unexpected '{k}'" for k in args if k not in props]
    types = {"string": str, "integer": int, "number": (int, float), "object": dict}
    for name, value in args.items():
        rule = props.get(name, {})
        if "type" in rule and not isinstance(value, types[rule["type"]]):
            problems.append(f"'{name}' should be {rule['type']}")
            continue
        if "enum" in rule and value not in rule["enum"]:
            problems.append(f"'{name}' must be one of {rule['enum']}")
        if "minimum" in rule and value < rule["minimum"]:
            problems.append(f"'{name}' below minimum {rule['minimum']}")
        if "pattern" in rule and not re.fullmatch(rule["pattern"], str(value)):
            problems.append(f"'{name}' doesn't match {rule['pattern']}")
    return problems

print(validate({"x": 1}, {"properties": {"x": {"type": "string"}}, "required": ["x", "y"]}))

## The generated tool set

In [ ]:
# --- GENERATED TOOLS (review this code — is it correct?) ---
GENERATED_TOOLS = [
    {"name": "get_customer", "description": "Gets a customer.",
     "input_schema": {"type": "object", "properties": {"name": {"type": "string"}}}},
    {"name": "get_invoices", "description": "Gets invoices from the billing API.",
     "input_schema": {"type": "object", "properties": {"customer_id": {"type": "string"}}}},
    {"name": "refund", "description": "Refund tool.",
     "input_schema": {"type": "object", "properties": {"customer": {"type": "string"},
                                                        "amount": {"type": "number"}}}},
    {"name": "send_email", "description": "Sends an email.",
     "input_schema": {"type": "object", "properties": {"to": {"type": "string"},
                                                        "body": {"type": "string"}}}},
]

def get_invoices(customer_id):          # the generated implementation: return whatever the API returns
    return BILLING_API

def refund(customer, amount):
    if amount > 480000:
        return "400 Bad Request"
    return "OK"

# Refund calls recorded from last week's traces (what the model actually sent)
RECORDED_REFUND_CALLS = [
    {"customer": "Acme Corp", "amount": 48},
    {"customer": "C-1042", "amount": 4800.0},
    {"customer": "ap@acme.example", "amount": 360000},
    {"customer": "C-1042"},
    {"invoice_id": "INV-2291", "amount_cents": 360000, "reason_code": "cancellation",
     "idempotency_key": "T-5512-1"},
]
for call in RECORDED_REFUND_CALLS:
    print(validate(call, GENERATED_TOOLS[2]["input_schema"]) or "accepted", "<-", call)

**Question 1.** All five recorded calls are accepted by the generated schema — including the one well-formed call, which passes only because its fields are never checked. For each of the four generated tools, name the most likely way the model will misuse it, based only on the four surfaces it can see.

In [ ]:
# Your review of each tool (as comments):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

- **get_customer(name)** — lookup by *name* is ambiguous ("Acme", "Acme Corp", "ACME Ltd") and can return the wrong customer or several; the description says nothing about what it returns. Look up by ID from the ticket, or return candidates with IDs and require a choice.
- **get_invoices(customer_id)** — the schema is fine-ish (nothing required), but the *return* is the raw API payload: the credit note that reduces INV-2291 sits in a different list, so the model must spot the link and subtract.
- **refund(customer, amount)** — refunds apply to invoices, not customers; the amount has no unit (48 vs 4800.0 vs 360000 were all sent); nothing is required; no idempotency key; the description doesn't say it's irreversible or what must be checked first. The only well-formed call is "accepted" merely because the schema has no `additionalProperties: false` — its fields are ignored.
- **send_email(to, body)** — the model chooses any recipient; for a refund agent that should be a reply to the ticket with the recipient fixed by the harness (Lesson 11).

</details>

**Question 2.** Write `ISSUE_REFUND_SCHEMA`: invoice ID with a pattern, integer `amount_cents` with a minimum, a `reason_code` enum, an `idempotency_key`, everything required, and `additionalProperties: false`. Re-run the recorded calls against it. How many are rejected, and for what?

In [ ]:
# Work here: ISSUE_REFUND_SCHEMA and the re-run

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
ISSUE_REFUND_SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string", "pattern": "INV-[0-9]{4,}"},
        "amount_cents": {"type": "integer", "minimum": 1},
        "reason_code": {"type": "string",
                        "enum": ["duplicate", "service_failure", "cancellation", "goodwill"]},
        "idempotency_key": {"type": "string"},
    },
    "required": ["invoice_id", "amount_cents", "reason_code", "idempotency_key"],
    "additionalProperties": False,
}
for call in RECORDED_REFUND_CALLS:
    print(validate(call, ISSUE_REFUND_SCHEMA) or "accepted", "<-", call)
```

The four ambiguous calls are rejected (missing invoice, no unit, unexpected `customer`/`amount`); only the well-formed call is accepted. A schema that rejects ambiguous calls turns a silent wrong refund into an error the model can correct.

</details>

**Question 3.** Replace the generated `refund` with `issue_refund(invoice_id, amount_cents, reason_code, idempotency_key)` that computes the refundable amount from `BILLING_API` (amount paid minus credits on that invoice) and returns *actionable* structured errors: unknown invoice (say which tool returns valid IDs), amount over the refundable limit (say the limit and why), and a repeated idempotency key (return the original refund, don't pay twice).

In [ ]:
# Work here: issue_refund with actionable errors

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
REFUNDS = {}                                   # idempotency_key -> refund record

def refundable_cents(invoice_id):
    inv = next((i for i in BILLING_API["invoices"] if i["id"] == invoice_id), None)
    if inv is None:
        return None
    credited = sum(c["amount"] for c in BILLING_API["credit_notes"] if c["invoice"] == invoice_id)
    return inv["amount_paid"] - credited

def issue_refund(invoice_id, amount_cents, reason_code, idempotency_key):
    if idempotency_key in REFUNDS:
        return {"ok": True, "repeat": True, **REFUNDS[idempotency_key]}
    limit = refundable_cents(invoice_id)
    if limit is None:
        return {"ok": False, "error": "invoice_not_found", "retryable": False,
                "message": f"No invoice {invoice_id}. Call get_billing_context(customer_id) for valid IDs."}
    if amount_cents > limit:
        return {"ok": False, "error": "amount_exceeds_refundable", "retryable": True,
                "message": f"At most {limit} cents is refundable on {invoice_id}: "
                           "credits already applied reduce the refundable amount."}
    REFUNDS[idempotency_key] = {"refund_id": f"RF-{len(REFUNDS) + 1:04d}",
                                "invoice_id": invoice_id, "amount_cents": amount_cents}
    return {"ok": True, **REFUNDS[idempotency_key]}

print(issue_refund("INV-9999", 100, "duplicate", "k1")["message"])
print(issue_refund("INV-2291", 480000, "cancellation", "k2")["message"])
print(issue_refund("INV-2291", 360000, "cancellation", "k3"))
print(issue_refund("INV-2291", 360000, "cancellation", "k3"))   # retried: same refund, not a second one
```

</details>

**Question 4.** Write the task-shaped `get_billing_context(customer_id)`: one record per invoice with `paid_cents`, `credited_cents` and `refundable_cents`, plus the plan. Compare its token count with the raw payload. Then answer: which matters more for correctness — the smaller size, or something else?

In [ ]:
# Work here: get_billing_context and the comparison

<details>
<summary>🔑 Reveal answer — Question 4</summary>

```python
def get_billing_context(customer_id):
    invoices = []
    for inv in BILLING_API["invoices"]:
        credited = sum(c["amount"] for c in BILLING_API["credit_notes"] if c["invoice"] == inv["id"])
        invoices.append({"invoice_id": inv["id"], "paid_cents": inv["amount_paid"],
                         "credited_cents": credited, "refundable_cents": inv["amount_paid"] - credited,
                         "period": f'{inv["period_start"]} to {inv["period_end"]}'})
    return {"customer_id": customer_id, "plan": "Teams (annual), 15 seats since 2026-06-14",
            "invoices": invoices}

print("raw:", tokens(BILLING_API), "| shaped:", tokens(get_billing_context("C-1042")))
```

The shaped record is several times smaller, and that saving repeats on every later loop step. But the bigger gain is correctness: `refundable_cents` is now a field. In the raw payload the $1,200 credit lives in another list, and the model has to notice the link and subtract — something it will sometimes fail to do.

</details>

**Question 5 (judgement).** The assistant's next suggestion is to "improve coverage" by adding `list_credit_notes`, `list_subscription_events` and `get_refund_policy_document` (returns the 2,000-token policy). What do you tell it?

In [ ]:
# Your answer (as a comment):

<details>
<summary>🔑 Reveal answer — Question 5</summary>

Decline all three as proposed. The first two are API-shaped tools that push the join back onto the model — the information they'd add already belongs *inside* `get_billing_context`. The policy document makes the model re-derive a business rule on every run; write the policy as code in `check_refund_eligibility(invoice_id, reason_code)` returning `eligible`, `max_refund_cents` and the rule that applied. Fewer, task-shaped tools also mean fewer tokens per step and fewer wrong-tool choices.

</details>

## Summary

1. The model sees only a tool's name, description, input schema and _______.
2. An irreversible tool that can be retried needs an _______ key.
3. Business rules the model would otherwise re-derive from a document belong in _______.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **what it returns**
2. **idempotency**
3. **code (a task-shaped tool)**

</details>